# Unified 3DGS Augmentation Pipeline
**Mode A** — Synthesis (Zero123++) | **Mode B** — Restoration (ControlNet) | **Mode C** — ViewCrafter

---

## Run order
1. **Cell 1** — Mount Drive
2. **Cell 2** — Set `SCENE_NAME` and `PIPELINE_MODE` (only cell you need to edit)
3. **Cell 3** — Common deps
4. **Cell 4** — Mode A/B setup *(skip if Mode C)*
5. **Cell 5a** — ViewCrafter clone + condacolab *(Mode C only — runtime restarts after this)*
6. **Cell 5b** — Re-mount Drive + re-run Cell 2 + create conda env *(Mode C only, after restart)*
7. **Cell 5c** — Download ViewCrafter model checkpoints *(Mode C only)*
8. **Cell 6** — CUDA submodules + COLMAP *(always)*
9. **Cell 7** — Launch Gradio UI — use the browser interface to run augmentation
10. **Cell 8** → **Cell 9** → **Cell 10** — SfM → Train → Metrics *(run manually after Gradio finishes)*

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 2: CONFIG — only cell you need to edit
import os

# ============================================================
# USER CONFIG
# ============================================================
SCENE_NAME     = "hotdog"      # folder name inside output_train/
PIPELINE_MODE  = "synthesis"   # "synthesis" | "restoration" | "viewcrafter"
RESTORE_PROMPT = "high quality photo, detailed, sharp focus, 8k"

# ── Derived paths — do not edit ──────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project"
GS_BASE    = "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting"

rawdata_path   = f"{DRIVE_BASE}/output_train/{SCENE_NAME}"
processed_path = f"{DRIVE_BASE}/output_processed/{SCENE_NAME}"
filename       = SCENE_NAME

os.makedirs(processed_path, exist_ok=True)
%cd "{DRIVE_BASE}"

print(f"Scene : {SCENE_NAME}")
print(f"Mode  : {PIPELINE_MODE}")
print(f"Input : {rawdata_path}")

In [ ]:
# Cell 3: Common deps
# Run this on first boot AND again after a condacolab restart (Mode C only).

import os, re, sys, tempfile

requirements = f"{DRIVE_BASE}/requirements.txt"

# ── Step 1: Filter requirements.txt ────────────────────────────────────────
# These packages have old version pins that conflict with Colab 2025 defaults.
# We skip them here and install at safe versions below.
_MANAGED = {
    'numpy',
    'huggingface-hub', 'huggingface_hub',
    'diffusers', 'transformers', 'peft',
    'accelerate',   # peft's transitive dep would pull in 1.x (circular import bug)
}

with open(requirements) as _f:
    _lines = _f.readlines()

_filtered, _skipped = [], []
for _line in _lines:
    _s = _line.strip()
    if not _s or _s.startswith('#'):
        _filtered.append(_line)
        continue
    _pkg = re.split(r'[=<>!\[;@\s]', _s)[0].lower().replace('_', '-')
    if _pkg in _MANAGED:
        _skipped.append(_s)
    else:
        _filtered.append(_line)

print("Skipped from requirements.txt (safe versions below):")
for _s in _skipped:
    print(f"  {_s}")

with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as _tmp:
    _tmp.writelines(_filtered)
    _fp = _tmp.name

!pip install -r "{_fp}" -q
os.unlink(_fp)

# ── Step 2a: numpy + HF ecosystem ──────────────────────────────────────────
!pip install -q \
    "numpy>=2.0" \
    "huggingface_hub>=0.33.5" \
    "diffusers>=0.27.2" \
    "transformers>=4.41.0" \
    "peft>=0.17.0" \
    gradio

# ── Step 2b: Pin accelerate AFTER peft ─────────────────────────────────────
# peft's dependency resolver pulls in accelerate 1.x which has a circular
# import bug in big_modeling.  Installing accelerate LAST overrides it.
# 0.34.x = last stable 0.x series: has clear_device_cache, no circular import.
!pip install -q "accelerate>=0.31.0,<1.0.0"

# ── Step 3: Patch clear_device_cache (fallback if 0.31–0.34 doesn't have it)
# Flush stale sys.modules cache first — without this, inspect.getfile()
# returns the path of a previously-loaded version, causing a false "OK".
for _k in list(sys.modules.keys()):
    if _k.startswith('accelerate'):
        del sys.modules[_k]

import importlib, inspect
import accelerate.utils.memory as _accel_mem
_accel_path = inspect.getfile(_accel_mem)
with open(_accel_path) as _f:
    _src = _f.read()
if 'clear_device_cache' not in _src:
    with open(_accel_path, 'a') as _f:
        _f.write("""

def clear_device_cache():
    \"\"\"Compatibility shim — injected by unified_pipeline Cell 3.\"\"\"
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
""")
    importlib.reload(_accel_mem)   # make in-process module reflect the patch
    print(f"Patched clear_device_cache -> {_accel_path}")
else:
    print("accelerate.utils.memory.clear_device_cache OK")

# ── Step 4: Diagnostics ─────────────────────────────────────────────────────
import numpy, accelerate, diffusers, peft, huggingface_hub
print(f"numpy            {numpy.__version__}")
print(f"huggingface_hub  {huggingface_hub.__version__}")
print(f"diffusers        {diffusers.__version__}")
print(f"accelerate       {accelerate.__version__}")
print(f"peft             {peft.__version__}")

import torch
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}")
print("Common deps ready")

In [ ]:
# Cell 4: Mode A/B setup — HuggingFace login + basicsr patch
# Skipped automatically if PIPELINE_MODE == "viewcrafter"
if PIPELINE_MODE in ["synthesis", "restoration"]:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    !sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' \
        /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
    print("HuggingFace login OK")
    print("basicsr compatibility patch applied")
else:
    print(f"Skipping Mode A/B setup (PIPELINE_MODE = '{PIPELINE_MODE}')")

---
## Cells 5a–5c: ViewCrafter setup (Mode C only)
**Skip these entirely if using Mode A or B.**

**Cell 5a** installs condacolab — the Colab runtime **restarts automatically** after it runs.

After the restart, run these cells in order:
1. **Cell 1** — Re-mount Drive
2. **Cell 2** — Re-run CONFIG (set `PIPELINE_MODE = "viewcrafter"` again)
3. **Cell 3** — Re-run common deps ⚠️ *the restart wipes pip installs including gradio — must reinstall*
4. **Cell 5a-post** — Completes ViewCrafter env setup (clone repo + create conda env)
5. **Cell 5c** — Download model checkpoints
6. Continue from **Cell 6**

In [ ]:
# Cell 5a: Clone ViewCrafter + install condacolab
# MODE C ONLY — runtime restarts after this cell
if PIPELINE_MODE == "viewcrafter":
    if not os.path.exists('/content/ViewCrafter'):
        !git clone https://github.com/Drexubery/ViewCrafter /content/ViewCrafter
    print("ViewCrafter repo ready")

    print("\nInstalling condacolab — runtime will restart automatically...")
    !pip install condacolab -q
    import condacolab
    condacolab.install()  # <-- triggers automatic restart
else:
    print(f"Skipping Cell 5a (PIPELINE_MODE = '{PIPELINE_MODE}')")

In [ ]:
# Cell 5a-post: [RUN THIS IMMEDIATELY AFTER THE RESTART — Mode C only]
# After condacolab restarts the runtime, all /content/ files are gone.
# This cell re-clones ViewCrafter and creates the conda environment.
# Before running this: re-run Cell 1 (Mount Drive), Cell 2 (CONFIG), Cell 3 (common deps).

if PIPELINE_MODE == "viewcrafter":
    # Re-clone ViewCrafter since /content/ was wiped by the restart
    if not os.path.exists('/content/ViewCrafter'):
        print("Re-cloning ViewCrafter after restart...")
        !git clone https://github.com/Drexubery/ViewCrafter /content/ViewCrafter
    else:
        print("ViewCrafter repo already present")

    # Create the isolated conda environment for ViewCrafter
    # This env gets its own torch (2.1.0) — completely separate from the base Python torch
    print("\nCreating viewcrafter_env (5-10 min)...")
    !conda create -n viewcrafter_env python=3.10 -y -q
    !conda install -n viewcrafter_env \
        -c pytorch -c nvidia -c pytorch3d \
        pytorch=2.1.0 torchvision pytorch-cuda=12.1 pytorch3d \
        -y -q
    # numpy<2.0 required — PyTorch 2.1.0 was released before numpy 2.0 and
    # betas.numpy() calls fail with RuntimeError: Numpy is not available on numpy 2.x
    # xformers==0.0.22.post7 matches PyTorch 2.1.0 — reduces ViewCrafter VRAM from
    # 31+ GiB (standard attention) to ~23 GiB (memory-efficient attention)
    !conda run -n viewcrafter_env pip install -q \
        av einops imageio imageio-ffmpeg kornia matplotlib moviepy \
        "numpy<2.0" open-clip-torch opencv-python Pillow pytorch-lightning \
        PyYAML roma scikit-image scikit-learn scipy tensorboard \
        timm tqdm "transformers<4.40.0" trimesh omegaconf \
        "xformers==0.0.22.post7"
    print("\nviewcrafter_env ready — run Cell 5c next (model downloads)")
else:
    print(f"Skipping (PIPELINE_MODE = '{PIPELINE_MODE}')")

In [ ]:
# Cell 5c: Download ViewCrafter model checkpoints (~25GB total)
if PIPELINE_MODE == "viewcrafter":
    import os
    os.makedirs("/content/ViewCrafter/checkpoints", exist_ok=True)

    dust3r = "/content/ViewCrafter/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth"
    vc_ckpt = "/content/ViewCrafter/checkpoints/model_sparse.ckpt"

    if not os.path.exists(dust3r):
        print("Downloading DUSt3R backbone...")
        !wget -q https://download.europe.naverlabs.com/ComputerVision/DUSt3R/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth \
             -P /content/ViewCrafter/checkpoints/
    else:
        print("DUSt3R already present")

    if not os.path.exists(vc_ckpt):
        print("Downloading ViewCrafter sparse model (~23GB)...")
        !wget -q https://huggingface.co/Drexubery/ViewCrafter_25_sparse/resolve/main/model_sparse.ckpt \
             -P /content/ViewCrafter/checkpoints/
    else:
        print("ViewCrafter checkpoint already present")

    print("All checkpoints ready")
else:
    print(f"Skipping Cell 5c (PIPELINE_MODE = '{PIPELINE_MODE}')")

In [ ]:
# Cell 6: CUDA submodules + COLMAP (always run regardless of mode)
import os, shutil
repo_path = GS_BASE

!pip install -q plyfile pyiqa
!pip uninstall -y tensorflow tensorflow-probability > /dev/null 2>&1
print("TF removed — GPU memory freed")

print("Compiling CUDA extensions on local SSD...")
!rm -rf /content/submodules_local
!cp -r "{repo_path}/submodules" /content/submodules_local
!rm -rf /content/submodules_local/diff-gaussian-rasterization/build
!rm -rf /content/submodules_local/diff-gaussian-rasterization/*.egg-info
!rm -rf /content/submodules_local/simple-knn/build
!rm -rf /content/submodules_local/simple-knn/*.egg-info
!pip install -q /content/submodules_local/diff-gaussian-rasterization
!pip install -q /content/submodules_local/simple-knn
print("CUDA extensions compiled")

print("Installing COLMAP, ffmpeg, xvfb...")
!sudo apt-get install -y ffmpeg colmap xvfb imagemagick > /dev/null 2>&1
!pip install -q pycolmap
print("All infrastructure ready — launch Gradio UI (Cell 7) next")

In [ ]:
# Cell 7: Gradio UI — unified interface for all three pipeline modes
import gradio as gr
import subprocess, os, shutil, glob
from PIL import Image

# Ensure optional deps are present (safe to re-run even if already installed)
subprocess.run(["pip", "install", "-q", "pillow-heif", "pyiqa"], capture_output=True)

# Path constants (match Cell 2)
_DRIVE_BASE = "/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project"
_GS_BASE    = "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting"
_VC_PYTHON  = "/usr/local/envs/viewcrafter_env/bin/python"
_VC_DIR     = "/content/ViewCrafter"


# ── Shared helpers ────────────────────────────────────────────────────────

def _normalise_images(directory, max_px=1920):
    """Convert unsupported formats to JPEG and resize oversized images.
    Operates on files already in `directory` — call AFTER copying to processed folder.
    Primary: pillow-heif. Fallback: ImageMagick (installed in Cell 6)."""
    _heic_via_pil = False
    try:
        import pillow_heif
        pillow_heif.register_heif_opener()
        _heic_via_pil = True
    except Exception:
        pass

    SUPPORTED = {'.jpg', '.jpeg', '.png', '.webp',
                 '.bmp', '.tiff', '.tif', '.heic', '.heif'}
    converted = resized = skipped = 0

    def _imagemagick_convert(src, dst):
        r = subprocess.run(['convert', src, dst], capture_output=True)
        return r.returncode == 0

    for fname in os.listdir(directory):
        ext  = os.path.splitext(fname)[1].lower()
        fpath = os.path.join(directory, fname)
        if not os.path.isfile(fpath) or ext not in SUPPORTED:
            continue
        new_path = os.path.splitext(fpath)[0] + '.jpg'
        is_heic  = ext in {'.heic', '.heif'}
        try:
            if is_heic and not _heic_via_pil:
                raise Exception("use ImageMagick")
            img = Image.open(fpath).convert('RGB')
            w, h = img.size
            changed = False
            if max(w, h) > max_px:
                scale = max_px / max(w, h)
                img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
                resized += 1
                changed = True
            if ext not in {'.jpg', '.jpeg'}:
                img.save(new_path, 'JPEG', quality=95)
                os.remove(fpath)
                converted += 1
            elif changed:
                img.save(fpath, 'JPEG', quality=95)
        except Exception:
            try:
                if _imagemagick_convert(fpath, new_path):
                    if fpath != new_path and os.path.exists(fpath):
                        os.remove(fpath)
                    img = Image.open(new_path)
                    w, h = img.size
                    if max(w, h) > max_px:
                        scale = max_px / max(w, h)
                        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
                        img.save(new_path, 'JPEG', quality=95)
                        resized += 1
                    converted += 1
                else:
                    skipped += 1
            except Exception as e:
                print(f"  Warning: could not convert {fname} — {e}")
                skipped += 1

    total = len([f for f in os.listdir(directory)
                 if os.path.splitext(f)[1].lower() in {'.jpg', '.jpeg', '.png'}])
    return converted, resized, skipped, total


def _copy_raw_to_processed(raw_dir, processed_dir):
    """Copy raw images from output_train into the processed folder.
    Originals on Drive are never modified — all normalisation happens on the copy."""
    os.makedirs(processed_dir, exist_ok=True)
    SUPPORTED = {'.jpg', '.jpeg', '.png', '.webp',
                 '.bmp', '.tiff', '.tif', '.heic', '.heif'}
    count = 0
    for fname in os.listdir(raw_dir):
        if os.path.splitext(fname)[1].lower() in SUPPORTED:
            shutil.copy2(os.path.join(raw_dir, fname),
                         os.path.join(processed_dir, fname))
            count += 1
    return count


def _screen(rawdata_path, processed_path):
    # process_file.py accepts "natural" or "synthetic"
    # synthesis→synthetic (generated images), restoration/viewcrafter→natural (real-world photos)
    _screen_mode = "synthetic" if PIPELINE_MODE == "synthesis" else "natural"
    # Override MPLBACKEND so matplotlib doesn't crash when inheriting the notebook env
    _env = {**os.environ, "MPLBACKEND": "Agg"}
    r = subprocess.run(
        ["python", "process_file.py", "--mode", _screen_mode,
         "--input_dir", rawdata_path, "--out_dir", processed_path],
        capture_output=True, text=True, cwd=_DRIVE_BASE, env=_env
    )
    return r.returncode == 0, r.stdout + ("\nERROR:\n" + r.stderr if r.returncode != 0 else "")


def _stage(scene_name, source_dir):
    dest = f"{_GS_BASE}/input_dataset/{scene_name}/input"
    os.makedirs(dest, exist_ok=True)
    count = 0
    for f in os.listdir(source_dir):
        if f.lower().endswith('.png'):
            shutil.copy2(os.path.join(source_dir, f), dest)
            count += 1
    return dest, count


def _preview(dest, n=16):
    """Preview PNG files (final staged output)."""
    pngs = sorted(glob.glob(f"{dest}/*.png"))[:n]
    return [Image.open(p) for p in pngs]


def _preview_any(directory, n=16):
    """Preview up to n images (JPG or PNG) from a directory.
    Used to show normalised inputs and quality-screened images."""
    exts = {'.jpg', '.jpeg', '.png'}
    files = sorted([f for f in os.listdir(directory)
                    if os.path.splitext(f)[1].lower() in exts])[:n]
    imgs = []
    for f in files:
        try:
            imgs.append(Image.open(os.path.join(directory, f)).convert('RGB'))
        except Exception:
            pass
    return imgs


def _save_scene(scene_name):
    import builtins
    builtins._gs_filename = scene_name


def _make_tv_replacement(indent):
    sp = ' ' * indent
    return (
        sp +
        'img = (lambda _a=np.array(pic, mode_to_nptype.get(pic.mode, np.uint8), copy=True):'
        ' torch.frombuffer(bytes(_a.tobytes()),'
        ' dtype={np.dtype("uint8"):torch.uint8,np.dtype("float32"):torch.float32,'
        'np.dtype("int32"):torch.int32}.get(_a.dtype,torch.uint8))'
        '.reshape(_a.shape).clone())()\n'
    )


def _patch_viewcrafter():
    msgs = []
    _diff_file = os.path.join(_VC_DIR, "lvdm/models/utils_diffusion.py")
    if os.path.exists(_diff_file):
        with open(_diff_file) as _f: _src = _f.read()
        if 'betas.numpy()' in _src:
            _src = _src.replace('return betas.numpy()',
                                'import numpy as _np; return _np.array(betas.detach().cpu().tolist())')
            with open(_diff_file, 'w') as _f: _f.write(_src)
            msgs.append("Patched utils_diffusion.py (betas.numpy() \u2192 tolist)")
        else:
            msgs.append("utils_diffusion.py already patched \u2014 OK")
    else:
        msgs.append("utils_diffusion.py not found \u2014 skip")

    _img_file = os.path.join(_VC_DIR, "extern/dust3r/dust3r/utils/image.py")
    if os.path.exists(_img_file):
        with open(_img_file) as _f: _src = _f.read()
        if 'Image.ANTIALIAS' in _src:
            _src = _src.replace('Image.ANTIALIAS', 'Image.LANCZOS')
            with open(_img_file, 'w') as _f: _f.write(_src)
            msgs.append("Patched dust3r/utils/image.py (ANTIALIAS \u2192 LANCZOS)")
        else:
            msgs.append("dust3r/utils/image.py already patched \u2014 OK")
    else:
        msgs.append("dust3r/utils/image.py not found \u2014 skip")

    _tv_file = ("/usr/local/envs/viewcrafter_env/lib/python3.10"
                "/site-packages/torchvision/transforms/functional.py")
    if not os.path.exists(_tv_file):
        msgs.append("torchvision functional.py not found \u2014 skip")
    else:
        with open(_tv_file) as _f: _lines = _f.readlines()
        _sentinel  = 'lambda _a=np.array(pic, mode_to_nptype'
        _original  = 'torch.from_numpy(np.array(pic, mode_to_nptype.get(pic.mode, np.uint8), copy=True))'
        _view_mark = 'img = img.view(pic.size[1]'
        if any(_sentinel in l for l in _lines):
            msgs.append("torchvision/transforms/functional.py already patched \u2014 OK")
        else:
            _new_lines = []; _patched = False
            for _line in _lines:
                if _original in _line:
                    _ind = len(_line) - len(_line.lstrip())
                    _new_lines.append(_make_tv_replacement(_ind)); _patched = True
                elif 'torch.frombuffer(_arr_tv' in _line:
                    _ind = len(_line) - len(_line.lstrip())
                    _new_lines.append(_make_tv_replacement(_ind)); _patched = True
                elif any(m in _line for m in ('import numpy as _np_tv', '_arr_tv = _np_tv.array', '_dtype_tv = ')):
                    pass
                elif _view_mark in _line and not _patched:
                    _ind = len(_line) - len(_line.lstrip())
                    _new_lines.append(_make_tv_replacement(_ind))
                    _new_lines.append(_line); _patched = True
                else:
                    _new_lines.append(_line)
            if _patched:
                with open(_tv_file, 'w') as _f: _f.writelines(_new_lines)
                msgs.append("Patched torchvision/transforms/functional.py (from_numpy \u2192 frombuffer)")
            else:
                msgs.append("torchvision/transforms/functional.py: no patch needed")

    _sc_file = ("/usr/local/envs/viewcrafter_env/lib/python3.10"
                "/site-packages/sitecustomize.py")
    _sc_sentinels = ["# torch.from_numpy frombuffer shim", "# torch/numpy interop shim v2"]
    if os.path.exists(_sc_file):
        with open(_sc_file) as _f: _sc = _f.read()
        if any(_s in _sc for _s in _sc_sentinels):
            _cut = len(_sc)
            for _s in _sc_sentinels:
                for _prefix in ("\n# " + _s.lstrip("# "), _s):
                    _idx = _sc.find(_prefix)
                    if 0 <= _idx < _cut: _cut = _idx
            with open(_sc_file, "w") as _f: _f.write(_sc[:_cut])
            msgs.append("Removed sitecustomize.py shim")
    _np_r = subprocess.run([_VC_PYTHON, "-c", "import numpy; print(numpy.__version__)"],
                           capture_output=True, text=True)
    _np_ver = _np_r.stdout.strip()
    if _np_ver:
        if int(_np_ver.split(".")[0]) >= 2:
            subprocess.run([_VC_PYTHON, "-m", "pip", "install", "-q", "numpy<2.0"],
                           capture_output=True)
            msgs.append(f"Downgraded numpy {_np_ver} \u2192 <2.0 in viewcrafter_env")
        else:
            msgs.append(f"numpy {_np_ver} in viewcrafter_env \u2014 OK")
    else:
        msgs.append("Could not check numpy version in viewcrafter_env")

    _tv_video_file = ("/usr/local/envs/viewcrafter_env/lib/python3.10"
                      "/site-packages/torchvision/io/video.py")
    if os.path.exists(_tv_video_file):
        with open(_tv_video_file) as _f: _src = _f.read()
        if 'frame.pict_type = "NONE"' in _src:
            _src = _src.replace('frame.pict_type = "NONE"', 'frame.pict_type = 0')
            with open(_tv_video_file, "w") as _f: _f.write(_src)
            msgs.append("Patched torchvision/io/video.py (pict_type \u201cNONE\u201d \u2192 0)")
        else:
            msgs.append("torchvision/io/video.py already patched \u2014 OK")
    else:
        msgs.append("torchvision/io/video.py not found \u2014 skip")

    _cond_file = os.path.join(_VC_DIR, "lvdm/modules/encoders/condition.py")
    if os.path.exists(_cond_file):
        with open(_cond_file) as _f: _src = _f.read()
        _old = "if self.model.visual.input_patchnorm:"
        _new = "if getattr(self.model.visual, 'input_patchnorm', False):"
        if _old in _src:
            with open(_cond_file, "w") as _f: _f.write(_src.replace(_old, _new))
            msgs.append("Patched condition.py (input_patchnorm \u2192 getattr fallback)")
        else:
            msgs.append("condition.py already patched \u2014 OK")
    else:
        msgs.append("condition.py not found \u2014 skip")

    _oc_r = subprocess.run([_VC_PYTHON, "-c", "import open_clip; print(open_clip.__version__)"],
                           capture_output=True, text=True)
    _oc_ver = _oc_r.stdout.strip()
    if _oc_ver:
        if [int(x) for x in _oc_ver.split(".")[:2]] >= [2, 23]:
            subprocess.run([_VC_PYTHON, "-m", "pip", "install", "-q", "open-clip-torch==2.20.0"],
                           capture_output=True)
            msgs.append(f"Downgraded open-clip-torch {_oc_ver} \u2192 2.20.0")
        else:
            msgs.append(f"open-clip-torch {_oc_ver} \u2014 OK")
    else:
        msgs.append("Could not check open-clip-torch version")
    return "\n".join(msgs)


# ── Mode A: Synthesis ─────────────────────────────────────────────────────

def run_synthesis(scene_name):
    logs = []
    def log(msg): logs.append(msg); return "\n".join(logs)
    rawdata   = f"{_DRIVE_BASE}/output_train/{scene_name}"
    processed = f"{_DRIVE_BASE}/output_processed/{scene_name}"
    os.makedirs(processed, exist_ok=True)
    if not os.path.exists(rawdata):
        yield log(f"ERROR: {rawdata} not found on Drive."), []; return
    yield log("[0/3] Normalising input images (format + resize)..."), []
    _conv, _rsz, _skp, _tot = _normalise_images(rawdata)
    yield log(f"  Converted: {_conv} | Resized: {_rsz} | Skipped: {_skp} | Ready: {_tot}"), []
    if _tot == 0:
        yield log("ERROR: No usable images after normalisation. Check file formats."), []; return
    yield log(f"[1/3] Running quality screener (process_file.py --mode {PIPELINE_MODE})..."), []
    ok, out = _screen(rawdata, processed)
    yield log(out), []
    if not ok: return
    _screened_imgs = _preview_any(processed)
    yield log(f"  {len(_screened_imgs)} images passed quality screen — previewing below."), _screened_imgs
    yield log("[2/3] Running Zero123++ synthesis (10-30 min)..."), []
    r = subprocess.run(
        ["python", "diffusion_script_v0.py",
         "--input_dir", processed, "--out_dir", processed, "--mode", "synthesis"],
        capture_output=True, text=True, cwd=_DRIVE_BASE
    )
    yield log(r.stdout[-3000:] + ("\nERROR:\n" + r.stderr[-500:] if r.returncode != 0 else "")), []
    if r.returncode != 0: return
    dest, count = _stage(scene_name, f"{processed}/final_{scene_name}_run")
    _save_scene(scene_name)
    yield log(f"[3/3] Done. {count} images staged.\nRun Cell 8 (convert_ai) next."), _preview(dest)


# ── Mode B: Restoration ───────────────────────────────────────────────────

def run_restoration(scene_name, prompt):
    logs = []
    def log(msg): logs.append(msg); return "\n".join(logs)
    rawdata   = f"{_DRIVE_BASE}/output_train/{scene_name}"
    processed = f"{_DRIVE_BASE}/output_processed/{scene_name}"
    os.makedirs(processed, exist_ok=True)
    if not os.path.exists(rawdata):
        yield log(f"ERROR: {rawdata} not found on Drive."), []; return
    yield log("[0/3] Normalising input images (format + resize)..."), []
    _conv, _rsz, _skp, _tot = _normalise_images(rawdata)
    yield log(f"  Converted: {_conv} | Resized: {_rsz} | Skipped: {_skp} | Ready: {_tot}"), []
    if _tot == 0:
        yield log("ERROR: No usable images after normalisation. Check file formats."), []; return
    yield log(f"[1/3] Running quality screener (process_file.py --mode {PIPELINE_MODE})..."), []
    ok, out = _screen(rawdata, processed)
    yield log(out), []
    if not ok: return
    _screened_imgs = _preview_any(processed)
    yield log(f"  {len(_screened_imgs)} images passed quality screen — previewing below."), _screened_imgs
    yield log("[2/3] Running ControlNet restoration (10-20 min)..."), []
    r = subprocess.run(
        ["python", "diffusion_script_v0.py",
         "--input_dir", processed, "--out_dir", processed,
         "--mode", "restoration", "--prompt", prompt],
        capture_output=True, text=True, cwd=_DRIVE_BASE
    )
    yield log(r.stdout[-3000:] + ("\nERROR:\n" + r.stderr[-500:] if r.returncode != 0 else "")), []
    if r.returncode != 0: return
    dest, count = _stage(scene_name, f"{processed}/final_{scene_name}_run")
    _save_scene(scene_name)
    yield log(f"[3/3] Done. {count} images staged.\nRun Cell 8 (convert_ai) next."), _preview(dest)


# ── Mode C: ViewCrafter ───────────────────────────────────────────────────

def run_viewcrafter(scene_name, video_length, ddim_steps):
    logs = []
    def log(msg): logs.append(msg); return "\n".join(logs)

    if not os.path.exists(_VC_PYTHON):
        yield log(f"ERROR: {_VC_PYTHON} not found.\nRun Cells 5a-5c first."), [], None; return

    raw_dir       = f"{_DRIVE_BASE}/output_train/{scene_name}/train"
    processed     = f"{_DRIVE_BASE}/output_processed/{scene_name}"
    processed_dir = f"{processed}/processed_{scene_name}"

    if not os.path.exists(raw_dir):
        yield log(f"ERROR: Input directory not found:\n  {raw_dir}"), [], None; return

    # Step 0: Copy raw images → processed folder, then normalise the copies.
    # Originals in output_train/ are never modified.
    yield log("[0/3] Copying raw images to processed folder..."), [], None
    n_copied = _copy_raw_to_processed(raw_dir, processed_dir)
    yield log(f"  Copied {n_copied} files to:\n  {processed_dir}"), [], None

    yield log("  Normalising (format conversion + resize)..."), [], None
    _conv, _rsz, _skp, _tot = _normalise_images(processed_dir)
    yield log(f"  Converted: {_conv} | Resized: {_rsz} | Skipped: {_skp} | Ready: {_tot}"), [], None
    if _tot == 0:
        yield log("ERROR: No usable images after normalisation.\n"
                  "Ensure ImageMagick is installed (Cell 6) and re-run."), [], None; return

    # Preview normalised inputs before quality screening
    _norm_preview = _preview_any(processed_dir)
    yield log(f"  {_tot} images normalised — previewing before quality screen."), _norm_preview, None

    # Quality screening — viewcrafter/restoration→"natural", synthesis→"synthetic"
    _screen_mode = "synthetic" if PIPELINE_MODE == "synthesis" else "natural"
    # Clean previous screened_dir so re-runs don't accumulate files from prior attempts
    screened_dir = os.path.join(processed_dir, "processed_train")
    if os.path.exists(screened_dir):
        shutil.rmtree(screened_dir)
        yield log("  Cleared stale processed_train/ from previous run."), [], None
    yield log(f"  Running quality screener (process_file.py --mode {_screen_mode})..."), [], None
    ok, out = _screen(processed_dir, processed_dir)
    yield log(out), [], None
    if not ok:
        yield log("ERROR: Quality screener failed. Check log above."), [], None; return
    # process_file.py writes passing images to out_dir/processed_train/
    # (screened_dir is already set above)
    _screened_imgs = _preview_any(screened_dir)
    yield log(f"  {len(_screened_imgs)} images passed quality screen — previewing below."), _screened_imgs, None

    # Rename images to numeric stems (0001.jpg …) — ViewCrafter sorts by int(stem)
    _img_files = sorted([f for f in os.listdir(screened_dir)
                         if os.path.splitext(f)[1].lower() in {'.jpg', '.jpeg', '.png'}])
    for _i, _fname in enumerate(_img_files):
        _ext = os.path.splitext(_fname)[1].lower()
        _new_name = f"{_i + 1:04d}{_ext}"
        if _fname != _new_name:
            os.rename(os.path.join(screened_dir, _fname),
                      os.path.join(screened_dir, _new_name))
    yield log(f"  Renamed {len(_img_files)} images to numeric filenames (0001–{len(_img_files):04d})."), [], None

    yield log(_patch_viewcrafter()), [], None

    output_dir = f"{processed}/final_{scene_name}_run"
    os.makedirs(output_dir, exist_ok=True)

    yield log(f"[1/3] Running ViewCrafter ({int(video_length)} frames, {int(ddim_steps)} steps)...\n"
              f"This takes 10-20 min."), [], None

    # Install av (PyAV) in viewcrafter_env — required by torchvision.io.write_video for saving output
    subprocess.run([_VC_PYTHON, "-m", "pip", "install", "-q", "av"], capture_output=True)
    _vc_env = {
        **os.environ,
        "MPLBACKEND": "Agg",
        "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:4096",
    }
    # Write stdout+stderr to Drive in real-time — survives Colab disconnections
    _vc_log_path = os.path.join(output_dir, "viewcrafter_run.log")
    yield log(f"  Live log → {_vc_log_path}\n  Check this file if you get disconnected."), [], None
    os.makedirs(output_dir, exist_ok=True)
    with open(_vc_log_path, "w", buffering=1) as _lf:
        r = subprocess.run([
            _VC_PYTHON, "inference.py",
            "--image_dir",    screened_dir, "--out_dir",       output_dir,
            "--mode",         "sparse_view_interp",
            "--bg_trd",       "0.2",         "--seed",         "123",
            "--ckpt_path",    "./checkpoints/model_sparse.ckpt",
            "--config",       "configs/inference_pvd_1024.yaml",
            "--ddim_steps",   str(int(ddim_steps)),
            "--video_length", str(int(video_length)),
            "--device",       "cuda:0",      "--height", "576", "--width", "1024",
            "--model_path",   "./checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth"
        ], stdout=_lf, stderr=subprocess.STDOUT, text=True, cwd=_VC_DIR, env=_vc_env)

    with open(_vc_log_path) as _lf:
        _vc_log_contents = _lf.read()
    out_text = _vc_log_contents[-3000:] if r.returncode == 0 else _vc_log_contents[-5000:]
    if r.returncode != 0:
        yield log(out_text), [], None; return
    yield log(out_text), [], None

    import glob as _glob
    found_mp4s = sorted(_glob.glob(os.path.join(output_dir, "**", "*.mp4"), recursive=True))
    if found_mp4s:
        render_mp4 = found_mp4s[-1]
        yield log(f"Found render.mp4 at: {render_mp4}"), [], render_mp4
    else:
        yield log(f"ERROR: render.mp4 not found under:\n  {output_dir}"), [], None; return

    yield log("[2/3] Extracting frames from render.mp4..."), [], render_mp4
    frames_dir = os.path.join(output_dir, "extracted_frames")
    os.makedirs(frames_dir, exist_ok=True)
    ff = subprocess.run([
        "ffmpeg", "-y", "-i", render_mp4,
        os.path.join(frames_dir, "frame_%04d.png")
    ], capture_output=True, text=True)
    if ff.returncode != 0:
        yield log(f"ffmpeg error:\n{ff.stderr[-1000:]}"), [], render_mp4; return

    extracted = sorted([f for f in os.listdir(frames_dir) if f.endswith('.png')])
    yield log(f"Extracted {len(extracted)} frames."), [], render_mp4

    dest = f"{_GS_BASE}/input_dataset/{scene_name}/input"
    os.makedirs(dest, exist_ok=True)
    yield log(f"[3/3] Staging to Drive:\n  {dest}"), [], render_mp4

    # Stage original (normalised) photos + ViewCrafter frames
    orig_count = 0
    for f in os.listdir(processed_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy2(os.path.join(processed_dir, f), dest)
            orig_count += 1
    for f in extracted:
        shutil.copy2(os.path.join(frames_dir, f), dest)

    total = len(os.listdir(dest))
    _save_scene(scene_name)
    yield log(
        f"[3/3] Done.\n"
        f"  Original photos : {orig_count}\n"
        f"  Extracted frames: {len(extracted)}\n"
        f"  Total for COLMAP: {total}\n"
        f"  Staged to: {dest}\n"
        f"Run Cell 8 (convert_ai) next."
    ), _preview(dest), render_mp4


# ── Build UI ──────────────────────────────────────────────────────────────
with gr.Blocks(title="3DGS Unified Pipeline") as demo:
    gr.Markdown(
        "## 3DGS Unified Pipeline\n"
        "**Before using:** Run Cells 1\u20136. "
        "**After Gradio finishes:** Run Cells 8\u201310 (SfM \u2192 Train \u2192 Metrics)."
    )
    with gr.Tabs():
        with gr.Tab("Mode A \u2014 Synthesis (Zero123++)"):
            gr.Markdown("_Object-centric NeRF datasets. Generates 6 novel views per anchor image._")
            a_scene = gr.Textbox(label="Scene Name", value="hotdog",
                                 placeholder="Must match a folder in output_train/")
            a_run = gr.Button("Run Synthesis Pipeline", variant="primary")
            a_log = gr.Textbox(label="Live Log", lines=14, interactive=False)
            a_gal = gr.Gallery(label="Output Preview", columns=4, height=400)
            a_run.click(fn=run_synthesis, inputs=[a_scene], outputs=[a_log, a_gal])
        with gr.Tab("Mode B \u2014 Restoration (ControlNet)"):
            gr.Markdown("_Natural or degraded images with blur/noise._")
            b_scene  = gr.Textbox(label="Scene Name", value="train",
                                  placeholder="Must match a folder in output_train/")
            b_prompt = gr.Textbox(label="ControlNet Prompt",
                                  value="high quality photo, detailed, sharp focus, 8k")
            b_run = gr.Button("Run Restoration Pipeline", variant="primary")
            b_log = gr.Textbox(label="Live Log", lines=14, interactive=False)
            b_gal = gr.Gallery(label="Output Preview", columns=4, height=400)
            b_run.click(fn=run_restoration, inputs=[b_scene, b_prompt], outputs=[b_log, b_gal])
        with gr.Tab("Mode C \u2014 ViewCrafter (Natural Scenes)"):
            gr.Markdown("_Natural scene video diffusion. Requires Cells 5a-5c._")
            c_scene = gr.Textbox(label="Scene Name", value="train",
                                 placeholder="Must match a folder in output_train/")
            with gr.Row():
                c_len   = gr.Slider(10, 50, value=25, step=5,  label="Video Length (frames)")
                c_steps = gr.Slider(20, 80, value=50, step=10, label="DDIM Steps")
            c_run   = gr.Button("Run ViewCrafter Pipeline", variant="primary")
            c_log   = gr.Textbox(label="Live Log", lines=14, interactive=False)
            c_video = gr.Video(label="render.mp4 Preview")
            c_gal   = gr.Gallery(label="Output Preview", columns=4, height=400)
            c_run.click(fn=run_viewcrafter, inputs=[c_scene, c_len, c_steps],
                        outputs=[c_log, c_gal, c_video])

demo.queue()
demo.launch(share=True)

---
## Manual Steps — Run after Gradio UI reports 'Done'
Cells 8–10 are always the same regardless of which mode you used.

In [ ]:
# Cell 8: Pose estimation — SuperPoint + LightGlue + COLMAP
import os, builtins
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# Pick up scene name set by Gradio; fall back to CONFIG value
filename = getattr(builtins, '_gs_filename', SCENE_NAME)
print(f"Processing scene: {filename}")

drive_path = f"{GS_BASE}/input_dataset/{filename}"
local_path = f"/content/local_workspace/{filename}"

print("Transferring dataset to local SSD...")
!mkdir -p "{local_path}"
!rm -rf "{local_path}/input" && mkdir -p "{local_path}/input"
!cp -r "{drive_path}/input/"* "{local_path}/input/"

print("Cleaning workspace...")
!rm -rf "{local_path}/sparse" "{local_path}/distorted"
!rm -f "{local_path}/database.db"

print("Running convert_ai.py (SuperPoint + LightGlue exhaustive matching)...")
%cd {GS_BASE}
!python convert_ai.py --source_path "{local_path}"

print("Syncing sparse/0 back to Drive...")
!cp -r "{local_path}"/* "{drive_path}/"
print(f"Done — {filename} ready for train.py")

In [ ]:
# Cell 9: 3DGS Training — 30,000 iterations
import shutil, os

LOCAL_INPUT  = f"/content/local_workspace/{filename}"
LOCAL_OUTPUT = f"/content/local_workspace/{filename}_final_run"
DRIVE_OUT    = f"{GS_BASE}/output/{filename}_final_run"

print("Staging COLMAP data for training...")
!rm -rf "{LOCAL_INPUT}" && mkdir -p "{LOCAL_INPUT}"
!cp -r "{GS_BASE}/input_dataset/{filename}/input" "{LOCAL_INPUT}/images"
!cp -r "{GS_BASE}/input_dataset/{filename}/sparse" "{LOCAL_INPUT}/"

print("Sanity check — sparse/0 contents:")
!ls -lh "{LOCAL_INPUT}/sparse/0"

print("Starting training (30k iterations, ~55 min)...")
%cd {GS_BASE}
!python train.py \
    -s "{LOCAL_INPUT}" \
    -m "{LOCAL_OUTPUT}" \
    --eval \
    --opacity_reset_interval 9000

print("Saving model to Drive...")
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUT, dirs_exist_ok=True)
print(f"Model saved to {DRIVE_OUT}")

In [ ]:
# Cell 10: Render held-out views + compute PSNR / SSIM / LPIPS
import os

LOCAL_INPUT = f"/content/local_workspace/{filename}"
DRIVE_OUT   = f"{GS_BASE}/output/{filename}_final_run"

print("Re-staging dataset for rendering...")
!rm -rf "{LOCAL_INPUT}" && mkdir -p "{LOCAL_INPUT}"
!cp -r "{GS_BASE}/input_dataset/{filename}/input" "{LOCAL_INPUT}/images"
!cp -r "{GS_BASE}/input_dataset/{filename}/sparse" "{LOCAL_INPUT}/"

%cd {GS_BASE}

print("Rendering test views...")
!python render.py \
    -m "{DRIVE_OUT}" \
    -s "{LOCAL_INPUT}" \
    --skip_train

print("Computing metrics (PSNR / SSIM / LPIPS)...")
!python metrics.py -m "{DRIVE_OUT}"